# Analysis for the Friedman benchmark

TODO Update with new SRB samples

The data sets are:

1. `f1a`: $f_1$ on 100 data points, no distractors
2. `f1b`: $f_1$ on 1000 data points, no distractors
3. `f1c`: $f_1$ on 100 data points, 5 columns of distractors
4. `f1d`: $f_1$ on 1000 data points, 5 columns of distractors

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

## Loading data

In [ ]:
f1expr = sympy.sympify("10*sin(π*x1*x2) + 20*(x3 - 1/2)^2 + 10*x4 + 5*x5")
f1expr

In [ ]:
sympy.expand(f1expr)

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report["sympy"] = full_report.expr_original_syms.apply(lambda e: au.parse_if_needed(e))
full_report["complexity"] = full_report.sympy.apply(lambda e: au.complexity(e))
full_report["sympy_defuzz"] = full_report.sympy.apply(lambda e: au.replace_near_integer(e.evalf()))
full_report["complexity_defuzz"] = full_report.sympy_defuzz.apply(lambda e: au.complexity(e))

In [ ]:
full_report.run_set.unique()

Make rows indexable by run set, data set, and sample number.

In [ ]:
fr1 = full_report.sort_values(["data_set", "mse"])
fr2 = fr1.set_index(["run_set", "data_set", "sample_num"])

In [ ]:
srb1 = fr2.loc["SRB-2026-06-25-1715-arr8"]
cht1 = fr2.loc["CHT-2026-06-25-1700"]
cht2 = fr2.loc[]

In [ ]:
srb1.groupby(level=["data_set"]).size()

These are the best ones overall

In [ ]:
srb1_min_mse_ixs = srb1.groupby(level=["data_set"]).mse.idxmin()
cht1_min_mse_ixs = cht1.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb1.loc[srb1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

The best for `f1a` and `f1b` are exactly correct up to fuzz.
For `f1c`, it found several correct terms, but with one cruft term involving distractors $x_6$ and $x_7$, and a wrong term involving $x_3$ that seems to be an approximation for $20(x_3 - 1/2)^2$ using $1 - \cos (x_3 - 1/2) \approx (1/2)(x_3 - 1/2)^2$.
For `f1d`, it's just lost.

In [ ]:
(srb1.loc[srb1_min_mse_ixs]
 .sympy_defuzz
 .apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2)))

In [ ]:
(srb1.loc[srb1_min_mse_ixs]
 .sympy_defuzz
 .apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2)))

In [ ]:
cht1_min_mse_ixs = cht1.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb1.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

In [ ]:
cht1.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

In [ ]:
au.count_by_threshold(srb1, threshold=1.0e-7)

Oddly, a lot of these have the correct terms, but they have a lot of extra cruft.
That means there's probably some numerical challenge with the process of solving for constants.

In [ ]:
srb1.loc["f1a", ["mse", "complexity_defuzz", "sympy_defuzz"]]

With generous defuzzing, we get 21 correct samples.

In [ ]:
srb1.loc["f1a"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
srb1.loc["f1b"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

Not much luck here:

In [ ]:
srb1.loc["f1c"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
srb1.loc["f1d"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
sns.displot(data=srb1,
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

Cheating does seem to help here:

In [ ]:
au.count_by_threshold(cht1, threshold=1.0e-7)

In [ ]:
sns.displot(data=cht1.sort_index(level=["data_set"]),
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

In [ ]:
cht1.loc["f1a"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
cht1.loc["f1c"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
cht1.loc["f1d"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))